# 🛸 Orbit Security // Autonomous Cloud Recon & Nuclei Sweeper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CmfH009/Orbit-Security/blob/main/tools/orbit_cloud_recon_colab.ipynb)

**Objective:** Leverage **100% Free Cloud Compute** (Google Colab Free T4 GPU / High-RAM CPU or Kaggle 30 hrs/week) to execute large-scale DNS permutation sweeps (50,000+ domains) and run 52 SaaS takeover Nuclei v3 templates with zero cloud token cost and zero CPU wear on your local workstation.

---

### Step 1: Initialize Cloud Environment & Install Tooling

In [ ]:
# Install high-performance DNS and network auditing libraries
!pip install --upgrade dnspython httpx pydantic pyyaml rich tqdm duckdb

# Install ProjectDiscovery Nuclei binary directly into cloud runner
!wget -q https://github.com/projectdiscovery/nuclei/releases/download/v3.3.7/nuclei_3.3.7_linux_amd64.zip -O /tmp/nuclei.zip
!unzip -q -o /tmp/nuclei.zip -d /usr/local/bin/
!nuclei -version
print("✓ Cloud runtime and Nuclei v3 installed successfully.")

### Step 2: Clone or Fetch Orbit Security Engine

In [ ]:
!git clone https://github.com/CmfH009/Orbit-Security.git /content/orbit-security || (cd /content/orbit-security && git pull)
%cd /content/orbit-security
!python -m pip install -e .
print("✓ Orbit Security loaded.")

### Step 3: Synthesize 52 Curated Nuclei v3 Verification Templates

In [ ]:
import os
from orbit_security.nuclei_generator import NucleiTemplateGenerator

generator = NucleiTemplateGenerator(output_dir="data/nuclei_templates")
written = generator.write_all_templates()
print(f"✓ Synthesized {len(written)} curated Nuclei v3 templates in cloud storage.")

### Step 4: Execute High-Speed Parallel DNS & Perimeter Audit (Free Cloud Network)

In [ ]:
# Load enrolled bug bounty targets
import json
with open("data/bounty_radar_state.json", "r") as f:
    radar = json.load(f)

programs = radar.get("programs", [])
domains = [p["domain"] for p in programs if "domain" in p]
print(f"Enrolled Targets ({len(domains)}):", domains[:10])

# Run Orbit Recon on enrolled domains
with open("/tmp/targets.txt", "w") as f:
    for d in domains:
        f.write(d + "\n")

!python tools/orbit-recon.py --targets-file /tmp/targets.txt --markdown /content/cloud_audit_report.md
print("✓ Cloud perimeter audit finished. Report written to /content/cloud_audit_report.md")

### Step 5: Execute Nuclei SaaS Takeover Verification

In [ ]:
# Run Nuclei with our curated templates
!nuclei -list /tmp/targets.txt -t data/nuclei_templates/ -severity low,medium,high,critical -jsonl -o /content/nuclei_findings.jsonl -silent || true

findings_count = 0
if os.path.exists("/content/nuclei_findings.jsonl"):
    with open("/content/nuclei_findings.jsonl") as f:
        findings = [json.loads(line) for line in f if line.strip()]
    findings_count = len(findings)
    print(f"✓ Nuclei sweep complete: {findings_count} findings detected.")
else:
    print("✓ No findings or targets clean.")

### Step 6: Download Audit Package to Local Workstation

In [ ]:
from google.colab import files
import tarfile

with tarfile.open("/content/orbit_cloud_audit_bundle.tar.gz", "w:gz") as tar:
    if os.path.exists("/content/cloud_audit_report.md"):
        tar.add("/content/cloud_audit_report.md", arcname="cloud_audit_report.md")
    if os.path.exists("/content/nuclei_findings.jsonl"):
        tar.add("/content/nuclei_findings.jsonl", arcname="nuclei_findings.jsonl")

print("Downloading audit bundle...")
files.download("/content/orbit_cloud_audit_bundle.tar.gz")